In [28]:

import os
import json
import zipfile
import shutil
import subprocess
from pathlib import Path

import pandas as pd




# Kaggle competition identifier
COMPETITION = "cmi-detect-behavior-with-sensor-data"

# Project directories
RAW_DIR = Path("../data/raw")                       # must contain ONLY train.csv and test.csv
PROCESSED_DIR = Path("../data/processed/cmi_sensor_data")

In [29]:
# Temporary directory for Kaggle downloads (will be deleted)
TMP_DIR = Path("../data/_kaggle_tmp")

# Files explicitly 
REQUIRED_FILES = ["train.csv", "test.csv"]

# Ensure required directories exist
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)



# Helper functions

def ensure_kaggle_cli():
    """
    Check that the Kaggle CLI is installed and accessible.
    """
    result = subprocess.run(["kaggle", "--version"], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(
            "Kaggle CLI not found. Install it using:\n"
            "pip install kaggle\n"
            "and configure your Kaggle API credentials."
        )
   # print("✅ Kaggle CLI available")
    


def download_competition_to_tmp(competition: str, out_dir: Path):
    """
    Download Kaggle competition files into a temporary directory.
    """
    print(f"⬇️ Downloading Kaggle data")
    cmd = ["kaggle", "competitions", "download", "-c", competition, "-p", str(out_dir)]
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("Kaggle download failed")

    print("✅ Download completed")

def unzip_all_files(directory: Path):
    """
    Unzip all .zip files found in the given directory.
    """
    zip_files = [p for p in directory.iterdir() if p.suffix.lower() == ".zip"]

    for zip_path in zip_files:
        #print(f"🗜️ Extracting: {zip_path.name}")
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(directory)

   # print("✅ All zip files extracted")


In [30]:
def clean_raw_directory(raw_dir: Path):
    """
    Remove everything from raw directory so only approved files remain.
    """
    for item in raw_dir.iterdir():
        if item.is_dir():
            shutil.rmtree(item)
        else:
            item.unlink()

   # print("🧹 Cleaned raw directory")

In [31]:
def copy_required_files(tmp_dir: Path, raw_dir: Path, required_files: list):
    """
    Copy only the professor-approved files from the temp directory into raw.
    """
    for filename in required_files:
        matches = list(tmp_dir.rglob(filename))
        if not matches:
            raise FileNotFoundError(f"Required file not found: {filename}")

        # Copy the first match found
        shutil.copy2(matches[0], raw_dir / filename)
        #print(f"✅ Copied {filename} into raw")

    #print("Raw directory now contains:", os.listdir(raw_dir))


In [32]:
def validate_raw_files(raw_dir: Path, required_files: list):
    """
    Ensure required raw files exist and are non-empty.
    """
    for filename in required_files:
        path = raw_dir / filename
        if not path.exists():
            raise FileNotFoundError(f"Missing required file: {filename}")
        if path.stat().st_size == 0:
            raise ValueError(f"{filename} is empty")

    

In [33]:
# 1. Ensure Kaggle CLI is available
ensure_kaggle_cli()

# 2. Download Kaggle dataset into temporary directory
download_competition_to_tmp(COMPETITION, TMP_DIR)

# 3. Extract downloaded zip files
unzip_all_files(TMP_DIR)

# 4. Clean raw directory so it contains only allowed files
clean_raw_directory(RAW_DIR)

# 5. Copy only train.csv and test.csv into raw directory
copy_required_files(TMP_DIR, RAW_DIR, REQUIRED_FILES)

# 6. Remove temporary directory (no zip or extra folders remain)
shutil.rmtree(TMP_DIR)
#print("🧼 Temporary Kaggle directory removed")

# 7. Validate raw files
validate_raw_files(RAW_DIR, REQUIRED_FILES)

# 8. Load raw datasets (no preprocessing, no splitting)
train_df = pd.read_csv(RAW_DIR / "train.csv")
test_df = pd.read_csv(RAW_DIR / "test.csv")

# Kaggle 
assert "gesture" in train_df.columns, "Target column 'gesture' must exist in train.csv"
assert "gesture" not in test_df.columns, "test.csv must not contain 'gesture'"

#print("✅ Verified: 'gesture' exists only in training data")

# 9. Save frozen raw copies for downstream stages
train_df.to_csv(PROCESSED_DIR / "train_raw.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "test_raw.csv", index=False)

# 10. Save dataset metadata
dataset_info = {
    "competition": COMPETITION,
    "raw_files_used": REQUIRED_FILES,
    "excluded_by_instruction": [
        "train_demographics.csv",
        "test_demographics.csv",
        "kaggle_evaluation/",
        "*.zip"
    ],
    "notes": (
        "No preprocessing or train-test split performed at ingestion. "
        "80:20 split will be applied after preprocessing in a sequence-aware manner."
    )
}

with open(PROCESSED_DIR / "dataset_info.json", "w") as f:
    json.dump(dataset_info, f, indent=2)

#print("Final raw contents:", os.listdir(RAW_DIR))
#print("Processed contents:", os.listdir(PROCESSED_DIR))

⬇️ Downloading Kaggle data
✅ Download completed
🗜️ Extracting: cmi-detect-behavior-with-sensor-data.zip
